## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


## Exploratory Data Analysis with Polars 

### Load CSV File into Polars DataFrame

In [4]:
df = pl.read_csv("data/Insurance_Date_data.csv",
                 ignore_errors=True)

df

Claim Number,Incident Date,Date Received
str,str,str
"""BCHKRDM32K""","""21-10-2007""","""31-10-2007"""
"""B3GPD5IZQW""","""26-05-2006""","""14-06-2006"""
"""EB757CV6XW""","""18-01-2004""","""10-02-2004"""
"""SP0Z0Q95OV""","""28-04-2004""","""06-05-2004"""
"""VKZUK7J3KK""","""04-11-2007""","""14-11-2007"""
…,…,…
"""AF9GJPNKEX""","""04-11-2004""","""18-11-2004"""
"""IB6C791V95""","""23-12-2005""","""23-02-2006"""
"""PGEDMDDHC2""","""07-09-2007""","""11-09-2007"""


In [5]:
column_data_types = zip(df.columns, df.dtypes)

for column, dtype in column_data_types:
    print(f"{column}:\t\t{dtype}")

Claim Number:		String
Incident Date:		String
Date Received:		String


### Display Summary Statistics for All Columns

In [7]:
summary = df.describe()
print(summary)

shape: (9, 4)
┌────────────┬──────────────┬───────────────┬───────────────┐
│ statistic  ┆ Claim Number ┆ Incident Date ┆ Date Received │
│ ---        ┆ ---          ┆ ---           ┆ ---           │
│ str        ┆ str          ┆ str           ┆ str           │
╞════════════╪══════════════╪═══════════════╪═══════════════╡
│ count      ┆ 34110        ┆ 34110         ┆ 34110         │
│ null_count ┆ 0            ┆ 0             ┆ 0             │
│ mean       ┆ null         ┆ null          ┆ null          │
│ std        ┆ null         ┆ null          ┆ null          │
│ min        ┆ 000TQE7Z0O   ┆ 01-01-2003    ┆ 01-01-2009    │
│ 25%        ┆ null         ┆ null          ┆ null          │
│ 50%        ┆ null         ┆ null          ┆ null          │
│ 75%        ┆ null         ┆ null          ┆ null          │
│ max        ┆ ZZW9JKTJMJ   ┆ 31-12-2008    ┆ 31-12-2009    │
└────────────┴──────────────┴───────────────┴───────────────┘


### Count of Nulls In Each Feature

In [8]:
for col in df.columns:
    print(f"{col} : {df[col].is_null().sum()}")

Claim Number : 0
Incident Date : 0
Date Received : 0


### Find Longest Text Length in Each Column

In [9]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

Claim Number,Incident Date,Date Received
u32,u32,u32
10,10,10


### Count Unique Values in Each Column

In [10]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

                 Unique values in Claim Number : 34110 
                Unique values in Incident Date : 2685  
                Unique values in Date Received : 1832  


### Retrieve Unique Values in Column If There Are Fewer Than 30

In [ ]:
for col in df.columns:
    unique_values = df[col].unique()
    if len(unique_values) < 30:
        print(f"'{col}' {len(unique_values)}: {unique_values.to_list()}")

### Check Distribution of Numerical Columns

In [12]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['id']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')